# Evaluation metrics (offline)

This notebook computes **accuracy, precision, recall, F1, PR AUC** (area under the precision–recall curve from `precision_recall_curve` + trapezoidal `auc`; and a few extras) from the local Langfuse exports next to `app.py`.

**Scope:** rows are restricted to the **latest calendar day** of score timestamps (`timestamp.max().date()`), i.e. the same **last snapshot / one batch** logic as `streamlit_app/app.py` (see Overview → last snapshot).

For the extra metrics we need the per-trace confusion label (`error_type`). Langfuse may put a numeric placeholder in `value` and the real label in `stringValue` — the notebook resolves both. If labels are missing, re-run:

```bash
python export_langfuse_csv.py
```

`accuracy` alone isn’t enough to derive F1/precision/recall without those labels.

**LaTeX:** run the export cell (after `metrics_by_model` is built) to write `metrics_export/` (or `streamlit_app/metrics_export/` when CSVs live under `streamlit_app/`): `metrics_wide.csv`, `metrics_long.csv`, `by_metric/<metric>.csv` and `.tsv`, `metrics_tabular.tex` (paste or `\input` the tabular), `metrics_macros.tex` (`\newcommand{\bench<Model><Metric>}{…}`), and `export_meta.json`.


In [1]:
from __future__ import annotations

import re
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    auc,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)

alt.data_transformers.disable_max_rows()

# Order for faceted charts (bar rows + daily small multiples)
METRIC_ORDER = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
    "balanced_accuracy",
    "pr_auc",
]

In [2]:
def _first_existing_path(*candidates: str) -> Path:
    for cand in candidates:
        p = Path(cand)
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these paths exist: {candidates}")


# Prefer streamlit_app/ first: a stray empty `langfuse_scores.csv` in the repo root would otherwise
# win over the real export next to this notebook (cwd-dependent footgun).
SCORES_CSV = _first_existing_path(
    "../streamlit_app/langfuse_scores.csv",
    "../langfuse_scores.csv",
)
TRACES_CSV = _first_existing_path(
    "../streamlit_app/langfuse_traces.csv",
    "../langfuse_traces.csv",
)

scores_raw = pd.read_csv(SCORES_CSV, low_memory=False)
traces_raw = pd.read_csv(TRACES_CSV, low_memory=False)

if scores_raw.empty or traces_raw.empty:
    raise ValueError(f"Loaded empty CSV — check paths: {SCORES_CSV=} {TRACES_CSV=}")

scores_raw.shape, traces_raw.shape

ParserError: Error tokenizing data. C error: Expected 7 fields in line 68113, saw 24


In [ ]:
def _pick_col(df: pd.DataFrame, *candidates: str) -> str:
    for cand in candidates:
        if cand in df.columns:
            return cand
    raise KeyError(f"None of these columns exist: {candidates}")


score_name_col = _pick_col(scores_raw, "name", "score_name")
trace_id_col = _pick_col(scores_raw, "traceId", "trace_id")
score_ts_col = _pick_col(scores_raw, "timestamp", "createdAt", "created_at")

trace_pk_col = _pick_col(traces_raw, "id", "trace_id")
trace_ts_col = _pick_col(traces_raw, "timestamp", "createdAt", "created_at")
trace_model_col = _pick_col(traces_raw, "metadata.model", "model")
trace_run_col = "metadata.run_id" if "metadata.run_id" in traces_raw.columns else ("run_id" if "run_id" in traces_raw.columns else None)

scores = scores_raw.copy()
scores["timestamp"] = pd.to_datetime(scores[score_ts_col], utc=True, errors="coerce")
scores = scores.dropna(subset=["timestamp"])

traces = traces_raw.copy()
traces["timestamp"] = pd.to_datetime(traces[trace_ts_col], utc=True, errors="coerce")
traces = traces.dropna(subset=["timestamp"])

traces_small = pd.DataFrame(
    {
        "trace_id": traces[trace_pk_col].astype(str).str.strip(),
        "model": traces[trace_model_col].fillna("unknown"),
        "run_id": traces[trace_run_col].fillna("") if trace_run_col else "",
        "trace_timestamp": traces["timestamp"],
    }
).drop_duplicates(subset=["trace_id"])

scores_small = scores[[trace_id_col, score_name_col, "timestamp"]].copy()
scores_small = scores_small.rename(columns={trace_id_col: "trace_id", score_name_col: "score_name"})
scores_small["trace_id"] = scores_small["trace_id"].astype(str).str.strip()

# keep all potentially relevant value columns for reconstruction
for col in ("value", "stringValue", "string_value", "valueString", "value_string", "comment"):
    if col in scores.columns:
        scores_small[col] = scores[col]
    else:
        scores_small[col] = np.nan

# Avoid mixed float/str inference in `value` (accuracy vs error_type) confusing downstream code.
if "value" in scores_small.columns:
    scores_small["value"] = scores_small["value"].astype("string")

df = scores_small.merge(traces_small, on="trace_id", how="left")
df = df[df["model"].notna() & (df["model"] != "unknown")].copy()

# Latest score day only — same batch as streamlit app "last snapshot" (max timestamp → calendar day).
_last_ts = df["timestamp"].max()
if pd.isna(_last_ts):
    raise ValueError("No valid score timestamps after merge.")
_last_day = pd.Timestamp(_last_ts).date()
df = df[df["timestamp"].dt.date == _last_day].copy()
if df.empty:
    raise ValueError(f"No score rows on last day {_last_day} — check CSV or timezone.")
print(f"Using last score day only: {_last_day}  ({len(df):,} score rows)")

df[["score_name", "model"]].value_counts().head(10)

In [ ]:
_COMMENT_RE = re.compile(r"expected\s*=\s*(True|False)\s*,\s*predicted\s*=\s*(True|False)")


def _parse_expected_predicted_from_comment(comment: str) -> tuple[bool, bool] | None:
    if not isinstance(comment, str):
        return None
    m = _COMMENT_RE.search(comment)
    if not m:
        return None
    expected = m.group(1) == "True"
    predicted = m.group(2) == "True"
    return expected, predicted


def _confusion_label_from_expected_predicted(expected: bool, predicted: bool) -> str:
    if expected is True and predicted is True:
        return "true_positive"
    if expected is True and predicted is False:
        return "false_negative"
    if expected is False and predicted is False:
        return "true_negative"
    return "false_positive"


_VALID_CONFUSION_LABELS = frozenset(
    {"true_positive", "false_positive", "true_negative", "false_negative"}
)


def _confusion_label_from_row(r: pd.Series) -> str | None:
    """Resolve error_type label. Langfuse often puts a numeric placeholder in `value` and the label in stringValue."""
    for col in ("stringValue", "string_value", "valueString", "value_string"):
        if col not in r.index:
            continue
        v = r[col]
        if pd.isna(v):
            continue
        s = str(v).strip().lower()
        if s in _VALID_CONFUSION_LABELS:
            return s
    v = r.get("value")
    if pd.notna(v):
        s = str(v).strip().lower()
        if s in _VALID_CONFUSION_LABELS:
            return s
    return None


def build_trace_level_table(df_scores_and_traces: pd.DataFrame) -> pd.DataFrame:
    acc = df_scores_and_traces[df_scores_and_traces["score_name"] == "accuracy"].copy()
    err = df_scores_and_traces[df_scores_and_traces["score_name"] == "error_type"].copy()

    if acc.empty:
        raise ValueError("No 'accuracy' scores found.")

    # 1 row per trace (accuracy)
    acc = acc.sort_values("timestamp").drop_duplicates(subset=["trace_id"], keep="last")
    acc["correct"] = pd.to_numeric(acc["value"], errors="coerce")
    acc["correct"] = acc["correct"].fillna(0.0).clip(0.0, 1.0).astype(int)

    trace_tbl = acc[["trace_id", "timestamp", "model", "run_id", "correct"]].copy()
    trace_tbl = trace_tbl.rename(columns={"timestamp": "score_timestamp"})

    if err.empty:
        trace_tbl["confusion_label"] = np.nan
        trace_tbl["y_true"] = np.nan
        trace_tbl["y_pred"] = np.nan
        return trace_tbl

    err = err.sort_values("timestamp").drop_duplicates(subset=["trace_id"], keep="last")

    def _label_row(r: pd.Series) -> str | None:
        label = _confusion_label_from_row(r)
        if label:
            return label
        parsed = _parse_expected_predicted_from_comment(r.get("comment"))
        if parsed:
            return _confusion_label_from_expected_predicted(*parsed)
        return None

    err["confusion_label"] = err.apply(_label_row, axis=1)

    err_small = err[["trace_id", "confusion_label"]]
    trace_tbl = trace_tbl.merge(err_small, on="trace_id", how="left")

    # Reconstruct y_true / y_pred from confusion label
    mapping = {
        "true_positive": (1, 1),
        "false_positive": (0, 1),
        "true_negative": (0, 0),
        "false_negative": (1, 0),
    }
    mapped = trace_tbl["confusion_label"].map(mapping)
    trace_tbl["y_true"] = mapped.map(lambda t: t[0] if isinstance(t, tuple) else np.nan)
    trace_tbl["y_pred"] = mapped.map(lambda t: t[1] if isinstance(t, tuple) else np.nan)

    return trace_tbl


traces_eval = build_trace_level_table(df)
traces_eval.head()

In [ ]:
def safe_metric(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception:
        return np.nan


def pr_auc_under_pr_curve(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Area under the precision–recall curve: trapezoidal integral ∫ precision d(recall)."""
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    return float(auc(recall, precision))


def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask].astype(int)
    y_pred = y_pred[mask].astype(int)

    if y_true.size == 0:
        return {
            "n": 0,
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "specificity": np.nan,
            "balanced_accuracy": np.nan,
            "pr_auc": np.nan,
        }

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    balanced_accuracy = np.nan
    if not np.isnan(specificity) and not np.isnan(sensitivity):
        balanced_accuracy = 0.5 * (specificity + sensitivity)

    # PR AUC = ∫ precision d(recall) on the curve from precision_recall_curve (trapezoidal rule via sklearn.metrics.auc).
    # y_score is predicted positive rate (here hard 0/1); with real probabilities the curve is richer.
    pr_auc = safe_metric(pr_auc_under_pr_curve, y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan

    return {
        "n": int(y_true.size),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "pr_auc": pr_auc,
    }


can_reconstruct = not traces_eval[["y_true", "y_pred"]].isna().all().all()
if not can_reconstruct:
    print(
        "Couldn't reconstruct y_true/y_pred from the current export.\n"
        "Re-run `python export_langfuse_csv.py` and ensure `error_type` is present with `value` in {true_positive,false_positive,true_negative,false_negative}.\n"
        "Showing accuracy-only until then."
    )

if can_reconstruct:
    # Avoid groupby.apply(Series): pandas 2.x can mis-shape the result so reindex() fills NaN everywhere.
    _rows: list[dict] = []
    for model, g in traces_eval.groupby("model", dropna=False):
        m = compute_binary_metrics(g["y_true"].to_numpy(), g["y_pred"].to_numpy())
        _rows.append({"model": model, **m})
    metrics_by_model = pd.DataFrame(_rows)
    expected_cols = [
        "model",
        "n",
        "tp",
        "fp",
        "tn",
        "fn",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "specificity",
        "balanced_accuracy",
        "pr_auc",
    ]
    metrics_by_model = metrics_by_model.reindex(columns=expected_cols)
else:
    metrics_by_model = (
        traces_eval.groupby(["model"], dropna=False)
        .agg(n=("trace_id", "count"), accuracy=("correct", "mean"))
        .reset_index()
    )
    for c in ("tp", "fp", "tn", "fn", "precision", "recall", "f1", "specificity", "balanced_accuracy", "pr_auc"):
        metrics_by_model[c] = np.nan

sort_metric = "f1" if can_reconstruct else "accuracy"
metrics_by_model.sort_values(sort_metric, ascending=False).head(20)

In [ ]:
from datetime import datetime, timezone
import json


def _metrics_export_dir() -> Path:
    if Path("langfuse_scores.csv").is_file():
        return Path("metrics_export")
    if Path("streamlit_app/langfuse_scores.csv").is_file():
        return Path("streamlit_app/metrics_export")
    return Path("metrics_export")


_export_root = _metrics_export_dir()
_export_root.mkdir(parents=True, exist_ok=True)
_by_metric = _export_root / "by_metric"
_by_metric.mkdir(parents=True, exist_ok=True)

_batch_day = pd.Timestamp(traces_eval["score_timestamp"].max()).date().isoformat()

# --- Wide / long CSV (import into LaTeX via csvsimple, pgfplots, pandas, etc.)
metrics_by_model.to_csv(_export_root / "metrics_wide.csv", index=False)

_long_cols = [c for c in METRIC_ORDER if c in metrics_by_model.columns]
if _long_cols:
    metrics_long_export = metrics_by_model.melt(
        id_vars=["model"],
        value_vars=_long_cols,
        var_name="metric",
        value_name="value",
    )
    metrics_long_export.to_csv(_export_root / "metrics_long.csv", index=False)

# --- One file per metric: model \\t value (TSV) + CSV
for m in _long_cols:
    sub = metrics_by_model[["model", m]].dropna().copy()
    sub.to_csv(_by_metric / f"{m}.csv", index=False)
    tsv_lines = [f"{r['model']}\t{r[m]:.12g}" for _, r in sub.iterrows()]
    (_by_metric / f"{m}.tsv").write_text("\n".join(tsv_lines) + "\n", encoding="utf-8")

# --- LaTeX: tabular only (wrap in your own table / longtable)
_tex_cols = [c for c in ["model", "n", "tp", "fp", "tn", "fn", *METRIC_ORDER] if c in metrics_by_model.columns]
_tex_df = metrics_by_model[_tex_cols].copy()
_latex_tabular = _tex_df.to_latex(index=False, escape=True, float_format="%.4f")
(_export_root / "metrics_tabular.tex").write_text(
    "% Auto-generated — last score day " + _batch_day + "\n"
    "% \\input{metrics_tabular.tex} inside your table environment\n"
    + _latex_tabular
    + "\n",
    encoding="utf-8",
)

# --- \\newcommand per (model, metric) for inline numbers: \\benchGPTFourOneNanofOne
_lines = ["% Auto-generated — " + _batch_day, "% Usage: \\bench<ModelAlnum><MetricAlnum>"]
for _, row in metrics_by_model.iterrows():
    mslug = "".join(c for c in str(row["model"]) if c.isalnum())
    for mc in _long_cols:
        v = row[mc]
        if pd.isna(v):
            continue
        mcslug = "".join(c for c in mc if c.isalnum())
        _lines.append(f"\\newcommand{{\\bench{mslug}{mcslug}}}{{{v:.6f}}}")
(_export_root / "metrics_macros.tex").write_text("\n".join(_lines) + "\n", encoding="utf-8")

_meta = {
    "last_score_day": _batch_day,
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_models": int(len(metrics_by_model)),
    "paths": {
        "metrics_wide_csv": str(_export_root / "metrics_wide.csv"),
        "metrics_long_csv": str(_export_root / "metrics_long.csv"),
        "metrics_tabular_tex": str(_export_root / "metrics_tabular.tex"),
        "metrics_macros_tex": str(_export_root / "metrics_macros.tex"),
        "by_metric_dir": str(_by_metric),
    },
}
(_export_root / "export_meta.json").write_text(json.dumps(_meta, indent=2), encoding="utf-8")

print("LaTeX-friendly export:", _export_root.resolve())

import shutil

_thesis_export = Path("../../hsmw-thesis-master/assets/metrics_export")
if _thesis_export.parent.is_dir():
    _thesis_export.mkdir(parents=True, exist_ok=True)
    for fname in ("metrics_tabular.tex", "metrics_macros.tex", "metrics_wide.csv", "export_meta.json"):
        src = _export_root / fname
        if src.is_file():
            shutil.copy2(str(src), str(_thesis_export / fname))
    if (_export_root / "metrics_long.csv").is_file():
        shutil.copy2(str(_export_root / "metrics_long.csv"), str(_thesis_export / "metrics_long.csv"))
    print("Thesis export:", _thesis_export.resolve())
else:
    print("Thesis assets dir not found — skipped:", _thesis_export)

In [ ]:
KEY_METRICS = ["accuracy", "precision", "recall", "f1"]

_sort_metric = "f1" if "f1" in metrics_by_model.columns and metrics_by_model["f1"].notna().any() else "accuracy"
_model_order = metrics_by_model.sort_values(_sort_metric, ascending=True)["model"].tolist()

# --- Chart A: key 4-metric comparison (thesis Section 6.5.3) ---
metrics_key = metrics_by_model.melt(
    id_vars=["model", "n", "tp", "fp", "tn", "fn"],
    value_vars=KEY_METRICS,
    var_name="metric",
    value_name="value",
)

chart_comparison = (
    alt.Chart(metrics_key)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None, scale=alt.Scale(domain=[0, 1])),
        y=alt.Y("model:N", title=None, sort=_model_order),
        color=alt.Color("model:N", legend=None),
        row=alt.Row("metric:N", title=None, sort=KEY_METRICS),
        tooltip=[
            alt.Tooltip("model:N"),
            alt.Tooltip("metric:N"),
            alt.Tooltip("value:Q", format=".4f"),
            alt.Tooltip("n:Q"),
        ],
    )
    .properties(width=700, height=alt.Step(18))
)

# --- Chart B: individual per-metric charts ---
charts_individual: dict[str, alt.Chart] = {}
for metric_key in METRIC_ORDER:
    if metric_key not in metrics_by_model.columns or metrics_by_model[metric_key].isna().all():
        continue
    sub = metrics_by_model[["model", metric_key, "n"]].dropna(subset=[metric_key]).copy()
    order = sub.sort_values(metric_key, ascending=True)["model"].tolist()
    c = (
        alt.Chart(sub)
        .mark_bar()
        .encode(
            x=alt.X(f"{metric_key}:Q", title=metric_key.replace("_", " ").title(), scale=alt.Scale(domain=[0, 1])),
            y=alt.Y("model:N", title=None, sort=order),
            color=alt.Color("model:N", legend=None),
            tooltip=[
                alt.Tooltip("model:N"),
                alt.Tooltip(f"{metric_key}:Q", format=".4f"),
                alt.Tooltip("n:Q"),
            ],
        )
        .properties(width=700, height=alt.Step(22), title=metric_key.replace("_", " ").title())
    )
    charts_individual[metric_key] = c

# --- Chart C: full faceted overview (all 7 metrics) ---
metrics_long = metrics_by_model.melt(
    id_vars=["model", "n", "tp", "fp", "tn", "fn"],
    value_vars=METRIC_ORDER,
    var_name="metric",
    value_name="value",
)

chart_all = (
    alt.Chart(metrics_long)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None, scale=alt.Scale(domain=[0, 1])),
        y=alt.Y("model:N", title=None, sort=_model_order),
        color=alt.Color("model:N", legend=None),
        row=alt.Row("metric:N", title=None, sort=METRIC_ORDER),
        tooltip=[
            alt.Tooltip("model:N"),
            alt.Tooltip("metric:N"),
            alt.Tooltip("value:Q", format=".4f"),
            alt.Tooltip("n:Q"),
            alt.Tooltip("tp:Q"),
            alt.Tooltip("fp:Q"),
            alt.Tooltip("tn:Q"),
            alt.Tooltip("fn:Q"),
        ],
    )
    .properties(width=700, height=alt.Step(18))
)

chart_comparison

In [ ]:
# Single calendar day in scope → per-model metrics are the bar chart above (no separate time series).
traces_eval["date"] = traces_eval["score_timestamp"].dt.floor("D")
print(
    "Date span in traces_eval:",
    traces_eval["score_timestamp"].min(),
    "→",
    traces_eval["score_timestamp"].max(),
)

In [ ]:
def _chart_dir() -> Path:
    if Path("langfuse_scores.csv").is_file():
        return Path("accuracy_charts")
    if Path("streamlit_app/langfuse_scores.csv").is_file():
        return Path("streamlit_app/accuracy_charts")
    return Path("accuracy_charts")


THESIS_ASSETS = Path("../../hsmw-thesis-master/assets")

_local_dir = _chart_dir()
_local_dir.mkdir(parents=True, exist_ok=True)
_hist = _local_dir / "history"
_hist.mkdir(parents=True, exist_ok=True)
_stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%SZ")

_all_charts: list[tuple[str, alt.Chart]] = [
    ("fig-metrics-comparison", chart_comparison),
    ("fig-metrics-all", chart_all),
]
for mk, ch in charts_individual.items():
    _all_charts.append((f"fig-metric-{mk}", ch))

try:
    for fname, chart in _all_charts:
        chart.save(str(_local_dir / f"{fname}.png"), scale_factor=2)
        chart.save(str(_hist / f"{fname}_{_stamp}.png"), scale_factor=2)
        if THESIS_ASSETS.is_dir():
            chart.save(str(THESIS_ASSETS / f"{fname}.png"), scale_factor=2)

    print("Local PNGs:", _local_dir.resolve())
    print("History:", _hist.resolve(), f"({_stamp})")
    if THESIS_ASSETS.is_dir():
        print("Thesis assets:", THESIS_ASSETS.resolve())
    else:
        print("Thesis assets dir not found — skipped:", THESIS_ASSETS)
except Exception as exc:
    print("PNG export failed:", exc)
    print("Install: pip install vl-convert-python")

In [ ]:
# Expected cost E_t[C](m) = C_t(m) + (C_FP * FP + C_FN * FN) / n
# C_FP and C_FN are operator parameters; default both to 1.0 USD.

C_FP = 1.0
C_FN = 1.0

cost_scores = df[df["score_name"] == "cost_usd"].copy()
cost_scores["cost_val"] = pd.to_numeric(cost_scores["value"], errors="coerce")

avg_cost_per_model = (
    cost_scores.groupby("model")["cost_val"]
    .mean()
    .reset_index()
    .rename(columns={"cost_val": "avg_inference_cost"})
)

expected_cost_df = metrics_by_model[["model", "n", "tp", "fp", "tn", "fn", "accuracy"]].copy()
expected_cost_df = expected_cost_df.merge(avg_cost_per_model, on="model", how="left")
expected_cost_df["avg_inference_cost"] = expected_cost_df["avg_inference_cost"].fillna(0.0)

expected_cost_df["expected_cost"] = (
    expected_cost_df["avg_inference_cost"]
    + (C_FP * expected_cost_df["fp"] + C_FN * expected_cost_df["fn"]) / expected_cost_df["n"]
)

expected_cost_df = expected_cost_df.sort_values("expected_cost", ascending=True)

# Export CSV
expected_cost_df.to_csv(_export_root / "expected_cost.csv", index=False)

# Export LaTeX tabular
_ec_tex_cols = ["model", "n", "fp", "fn", "accuracy", "avg_inference_cost", "expected_cost"]
_ec_tex = expected_cost_df[_ec_tex_cols].copy()
_ec_tex.columns = ["model", "n", "FP", "FN", "accuracy", "avg cost (USD)", "E[C] (USD)"]
_ec_latex = _ec_tex.to_latex(index=False, escape=True, float_format="%.6f")
(_export_root / "expected_cost_tabular.tex").write_text(
    "% Auto-generated — last score day " + _batch_day + "\n"
    f"% C_FP = {C_FP}, C_FN = {C_FN}\n"
    "% \\input{expected_cost_tabular.tex} inside your table environment\n"
    + _ec_latex
    + "\n",
    encoding="utf-8",
)

# Copy to thesis assets
if _thesis_export.is_dir():
    for fname in ("expected_cost.csv", "expected_cost_tabular.tex"):
        src = _export_root / fname
        if src.is_file():
            shutil.copy2(str(src), str(_thesis_export / fname))

print(f"Expected cost (C_FP={C_FP}, C_FN={C_FN}):")
expected_cost_df[_ec_tex_cols]